# Smartphone Addiction Prediction - Kaggle Playground Series S6E8

This notebook builds a reproducible binary-classification solution for the Kaggle competition.

The workflow is designed for Google Colab and focuses on:
- leakage-safe validation with stratified cross-validation;
- domain-inspired feature engineering;
- diverse tree-based models;
- out-of-fold predictions;
- rank-Gaussian blending and a logistic meta-model;
- a final Kaggle submission file.

The target is `addicted_label`, where 1 means the participant is classified as addicted according to the competition definition.


## Reproducibility contract

- Competition: Playground Series S6E8.
- Target column: `addicted_label`.
- Identifier column: `id`.
- Validation: 5-fold StratifiedKFold with a fixed seed.
- All preprocessing that learns from data is fitted on the training fold only.
- The test row order is preserved when creating the submission.
- Kaggle credentials are read from a Colab Secret named `KAGGLE_API_TOKEN`.
- Do not commit the Kaggle token, raw data, model binaries, or generated submissions to GitHub.


In [ ]:
!pip -q install -U kaggle lightgbm catboost xgboost


In [ ]:
import os
import json
import time
import zipfile
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import rankdata, norm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SEED = 42
TARGET = "addicted_label"
ID_COL = "id"
COMPETITION = "playground-series-s6e8"

# Use FAST_MODE=True to validate the notebook quickly before the full run.
FAST_MODE = False
RUN_XGB = True
N_FOLDS = 3 if FAST_MODE else 5
N_ITERATIONS = 350 if FAST_MODE else 1800

print("Configuration:", {
    "seed": SEED,
    "folds": N_FOLDS,
    "iterations": N_ITERATIONS,
    "run_xgb": RUN_XGB,
})


## Kaggle authentication

Create a Colab Secret called `KAGGLE_API_TOKEN`, enable notebook access for that secret, and run the next cell. The token is never printed.


In [ ]:
from google.colab import userdata

try:
    kaggle_token = userdata.get("KAGGLE_API_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Create a Colab Secret named KAGGLE_API_TOKEN, enable notebook access, and run this cell again."
    ) from exc

if not kaggle_token:
    raise RuntimeError("The KAGGLE_API_TOKEN secret is empty.")

os.environ["KAGGLE_API_TOKEN"] = kaggle_token
print("Kaggle authentication configured without printing the token.")


In [ ]:
DATA_DIR = Path("/content/s6e8_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "train.csv").exists():
    subprocess.run(
        ["kaggle", "competitions", "download", "-c", COMPETITION, "-p", str(DATA_DIR)],
        check=True,
    )

zip_files = sorted(DATA_DIR.glob("*.zip"))
if zip_files:
    with zipfile.ZipFile(zip_files[0]) as archive:
        archive.extractall(DATA_DIR)

required_files = ["train.csv", "test.csv", "sample_submission.csv"]
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing competition files: {missing_files}")

print("Data directory:", DATA_DIR)
print("Files:", sorted(path.name for path in DATA_DIR.iterdir()))


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)
display(train.head())


## Initial audit

Before modeling, inspect the target balance, data types, missing values, and possible identifier columns. The competition score is based on ranking predicted probabilities, so the notebook keeps probabilities instead of hard class labels.


In [ ]:
assert TARGET in train.columns, f"{TARGET} is not present in train.csv"
assert TARGET not in test.columns, f"{TARGET} unexpectedly appears in test.csv"
assert ID_COL in train.columns and ID_COL in test.columns

print("Target distribution:")
display(train[TARGET].value_counts(normalize=True).rename("share").to_frame())

schema = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum().reindex(train.columns, fill_value=0),
    "n_unique_train": train.nunique(dropna=False),
})
display(schema.sort_values(["missing_train", "n_unique_train"], ascending=[False, True]).head(30))


In [ ]:
numeric_columns = train.select_dtypes(include=np.number).columns.tolist()
numeric_columns = [column for column in numeric_columns if column != TARGET]

if numeric_columns:
    train[numeric_columns].hist(figsize=(16, 12), bins=30)
    plt.suptitle("Numeric feature distributions", y=1.02)
    plt.tight_layout()
    plt.show()


## Feature engineering

The raw columns describe daily phone usage, screen time, notifications, social activity, gaming, study/work, and sleep. In addition to the original variables, we create interpretable ratios and consistency features.

The function is deliberately defensive: it only creates a feature when the required source columns exist. This lets the notebook survive small schema changes without silently breaking.


In [ ]:
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator

def add_if_possible(frame, output_name, function, required_columns):
    if all(column in frame.columns for column in required_columns):
        frame[output_name] = function(frame)

def build_features(frame):
    result = frame.copy()

    result["missing_count"] = result.isna().sum(axis=1)

    add_if_possible(
        result,
        "screen_component_total",
        lambda df: df["daily_screen_time"] + df["social_media_time"] + df["gaming_time"],
        ["daily_screen_time", "social_media_time", "gaming_time"],
    )
    add_if_possible(
        result,
        "time_accounting_residual",
        lambda df: df["daily_screen_time"] - (
            df["social_media_time"] + df["gaming_time"] + df["work_study_time"]
        ),
        ["daily_screen_time", "social_media_time", "gaming_time", "work_study_time"],
    )
    add_if_possible(
        result,
        "weekend_lift",
        lambda df: df["weekend_screen_time"] - df["daily_screen_time"],
        ["weekend_screen_time", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "weekend_ratio",
        lambda df: safe_divide(df["weekend_screen_time"], df["daily_screen_time"]),
        ["weekend_screen_time", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "social_share_of_daily",
        lambda df: safe_divide(df["social_media_time"], df["daily_screen_time"]),
        ["social_media_time", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "gaming_share_of_daily",
        lambda df: safe_divide(df["gaming_time"], df["daily_screen_time"]),
        ["gaming_time", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "work_study_share_of_daily",
        lambda df: safe_divide(df["work_study_time"], df["daily_screen_time"]),
        ["work_study_time", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "sleep_minus_screen",
        lambda df: df["sleep_hours"] - df["daily_screen_time"],
        ["sleep_hours", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "screen_to_sleep_ratio",
        lambda df: safe_divide(df["daily_screen_time"], df["sleep_hours"]),
        ["daily_screen_time", "sleep_hours"],
    )
    add_if_possible(
        result,
        "opens_per_screen_hour",
        lambda df: safe_divide(df["phone_unlocks"], df["daily_screen_time"]),
        ["phone_unlocks", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "notifications_per_screen_hour",
        lambda df: safe_divide(df["notifications"], df["daily_screen_time"]),
        ["notifications", "daily_screen_time"],
    )
    add_if_possible(
        result,
        "notifications_per_open",
        lambda df: safe_divide(df["notifications"], df["phone_unlocks"]),
        ["notifications", "phone_unlocks"],
    )
    add_if_possible(
        result,
        "engagement_events",
        lambda df: df["phone_unlocks"] + df["notifications"],
        ["phone_unlocks", "notifications"],
    )

    for column in CAT_COLS:
        if column in result.columns:
            result[column] = result[column].astype("string").fillna("__MISSING__")

    result = result.replace([np.inf, -np.inf], np.nan)
    return result

train_features = build_features(train.drop(columns=[TARGET]))
test_features = build_features(test.copy())

# Keep the same columns and order in train and test.
test_features = test_features.reindex(columns=train_features.columns)
print("Feature matrix:", train_features.shape)
display(train_features.head())


## Shared cross-validation

The same folds are reused for every model. This makes model comparisons fair and gives us out-of-fold predictions for a leakage-safe blend.


In [ ]:
y = train[TARGET].astype(int).to_numpy()

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)
folds = list(skf.split(train_features, y))

print("Fold sizes:")
for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
    print(
        f"Fold {fold_id}: train={len(train_idx):,}, "
        f"valid={len(valid_idx):,}, "
        f"positive_rate={y[valid_idx].mean():.4f}"
    )


## Encoding for LightGBM and XGBoost

CatBoost can consume categorical columns directly. LightGBM and XGBoost use an ordinal encoding fitted on the full training feature matrix without using the target. Missing categories remain represented by an explicit category.


In [ ]:
categorical_columns = [
    column for column in CAT_COLS
    if column in train_features.columns
]
numeric_feature_columns = [
    column for column in train_features.columns
    if column not in categorical_columns
]

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
encoder.fit(train_features[categorical_columns].astype(str))

def make_tree_matrix(frame):
    numeric_part = frame[numeric_feature_columns].copy()
    categorical_part = pd.DataFrame(
        encoder.transform(frame[categorical_columns].astype(str)),
        columns=categorical_columns,
        index=frame.index,
    )
    return pd.concat([numeric_part, categorical_part], axis=1).astype(float)

X_tree = make_tree_matrix(train_features)
X_test_tree = make_tree_matrix(test_features)

CAT_INDICES = [
    train_features.columns.get_loc(column)
    for column in categorical_columns
]

X_cat = train_features.copy()
X_test_cat = test_features.copy()

print("Categorical columns:", categorical_columns)
print("Tree matrix:", X_tree.shape)
print("CatBoost categorical indices:", CAT_INDICES)


## CatBoost baseline

CatBoost is useful here because the dataset contains mixed numeric and categorical information. Early stopping selects the useful number of iterations inside each fold.


In [ ]:
def run_catboost_oof(X_train, X_test, y_values):
    oof = np.zeros(len(X_train), dtype=float)
    test_fold_predictions = []
    fold_scores = []

    for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
        model = CatBoostClassifier(
            iterations=N_ITERATIONS,
            depth=8,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=SEED + fold_id,
            l2_leaf_reg=5.0,
            random_strength=0.5,
            allow_writing_files=False,
            verbose=False,
            thread_count=-1,
        )

        model.fit(
            X_train.iloc[train_idx],
            y_values[train_idx],
            cat_features=CAT_INDICES,
            eval_set=(X_train.iloc[valid_idx], y_values[valid_idx]),
            use_best_model=True,
            early_stopping_rounds=120,
            verbose=False,
        )

        valid_prediction = model.predict_proba(X_train.iloc[valid_idx])[:, 1]
        test_prediction = model.predict_proba(X_test)[:, 1]

        oof[valid_idx] = valid_prediction
        test_fold_predictions.append(test_prediction)

        fold_auc = roc_auc_score(y_values[valid_idx], valid_prediction)
        fold_scores.append(fold_auc)
        print(f"CatBoost fold {fold_id}: AUC={fold_auc:.6f}, best_iteration={model.get_best_iteration()}")

    return oof, np.mean(test_fold_predictions, axis=0), fold_scores

cat_oof, cat_test, cat_fold_scores = run_catboost_oof(
    X_cat,
    X_test_cat,
    y,
)
print("CatBoost OOF AUC:", roc_auc_score(y, cat_oof))


## LightGBM model

LightGBM gives a different tree-growing strategy from CatBoost. Diversity between strong models is useful because their errors are not identical.


In [ ]:
def run_lightgbm_oof(X_train, X_test, y_values):
    oof = np.zeros(len(X_train), dtype=float)
    test_fold_predictions = []
    fold_scores = []

    for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
        model = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=N_ITERATIONS,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=80,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=SEED + fold_id,
            n_jobs=-1,
            verbosity=-1,
        )

        model.fit(
            X_train.iloc[train_idx],
            y_values[train_idx],
            eval_set=[(X_train.iloc[valid_idx], y_values[valid_idx])],
            callbacks=[
                lgb.early_stopping(120, verbose=False),
                lgb.log_evaluation(0),
            ],
        )

        valid_prediction = model.predict_proba(X_train.iloc[valid_idx])[:, 1]
        test_prediction = model.predict_proba(X_test)[:, 1]

        oof[valid_idx] = valid_prediction
        test_fold_predictions.append(test_prediction)

        fold_auc = roc_auc_score(y_values[valid_idx], valid_prediction)
        fold_scores.append(fold_auc)
        print(f"LightGBM fold {fold_id}: AUC={fold_auc:.6f}, best_iteration={model.best_iteration_}")

    return oof, np.mean(test_fold_predictions, axis=0), fold_scores

lgb_oof, lgb_test, lgb_fold_scores = run_lightgbm_oof(
    X_tree,
    X_test_tree,
    y,
)
print("LightGBM OOF AUC:", roc_auc_score(y, lgb_oof))


## Optional XGBoost model

XGBoost is enabled by default. Set `RUN_XGB = False` near the top if a Colab session is short on time or memory.


In [ ]:
def run_xgboost_oof(X_train, X_test, y_values):
    oof = np.zeros(len(X_train), dtype=float)
    test_fold_predictions = []
    fold_scores = []

    for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
        model = xgb.XGBClassifier(
            n_estimators=N_ITERATIONS,
            max_depth=7,
            learning_rate=0.03,
            min_child_weight=8,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=SEED + fold_id,
            n_jobs=-1,
        )

        model.fit(
            X_train.iloc[train_idx],
            y_values[train_idx],
            eval_set=[(X_train.iloc[valid_idx], y_values[valid_idx])],
            verbose=False,
        )

        valid_prediction = model.predict_proba(X_train.iloc[valid_idx])[:, 1]
        test_prediction = model.predict_proba(X_test)[:, 1]

        oof[valid_idx] = valid_prediction
        test_fold_predictions.append(test_prediction)

        fold_auc = roc_auc_score(y_values[valid_idx], valid_prediction)
        fold_scores.append(fold_auc)
        print(f"XGBoost fold {fold_id}: AUC={fold_auc:.6f}")

    return oof, np.mean(test_fold_predictions, axis=0), fold_scores

if RUN_XGB:
    xgb_oof, xgb_test, xgb_fold_scores = run_xgboost_oof(
        X_tree,
        X_test_tree,
        y,
    )
    print("XGBoost OOF AUC:", roc_auc_score(y, xgb_oof))
else:
    xgb_oof = None
    xgb_test = None
    xgb_fold_scores = []


## Out-of-fold model comparison


In [ ]:
oof_predictions = {
    "catboost": cat_oof,
    "lightgbm": lgb_oof,
}
test_predictions = {
    "catboost": cat_test,
    "lightgbm": lgb_test,
}

if RUN_XGB:
    oof_predictions["xgboost"] = xgb_oof
    test_predictions["xgboost"] = xgb_test

model_scores = pd.DataFrame([
    {
        "model": name,
        "oof_auc": roc_auc_score(y, prediction),
        "prediction_mean": prediction.mean(),
        "prediction_std": prediction.std(),
    }
    for name, prediction in oof_predictions.items()
]).sort_values("oof_auc", ascending=False)

display(model_scores)

oof_frame = pd.DataFrame(oof_predictions)
test_frame = pd.DataFrame(test_predictions)
display(oof_frame.head())


## Rank-Gaussian blending and stacking

AUC is invariant to any strictly increasing transformation of predictions. We therefore transform each model's predictions into a normalized rank scale before blending.

The meta-model is evaluated on out-of-fold predictions only. It never sees a validation target while the base model is being trained.


In [ ]:
def rank_gauss(values):
    values = np.asarray(values, dtype=float)
    ranks = (rankdata(values, method="average") - 0.5) / len(values)
    return norm.ppf(np.clip(ranks, 1e-6, 1 - 1e-6))

oof_meta = pd.DataFrame({
    name: rank_gauss(prediction)
    for name, prediction in oof_predictions.items()
})
test_meta = pd.DataFrame({
    name: rank_gauss(test_predictions[name])
    for name in oof_predictions
})

meta_oof = np.zeros(len(train_features), dtype=float)
meta_test_fold_predictions = []
meta_fold_scores = []

for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
    scaler = StandardScaler()
    X_meta_train = scaler.fit_transform(oof_meta.iloc[train_idx])
    X_meta_valid = scaler.transform(oof_meta.iloc[valid_idx])
    X_meta_test = scaler.transform(test_meta)

    meta_model = LogisticRegression(
        C=0.03,
        solver="lbfgs",
        max_iter=2000,
        random_state=SEED + fold_id,
    )
    meta_model.fit(X_meta_train, y[train_idx])

    valid_prediction = meta_model.predict_proba(X_meta_valid)[:, 1]
    test_prediction = meta_model.predict_proba(X_meta_test)[:, 1]

    meta_oof[valid_idx] = valid_prediction
    meta_test_fold_predictions.append(test_prediction)

    fold_auc = roc_auc_score(y[valid_idx], valid_prediction)
    meta_fold_scores.append(fold_auc)
    print(f"Stack fold {fold_id}: AUC={fold_auc:.6f}")

stack_test = np.mean(meta_test_fold_predictions, axis=0)
stack_auc = roc_auc_score(y, meta_oof)

print("Stack OOF AUC:", stack_auc)
print("Simple rank average OOF AUC:", roc_auc_score(y, oof_meta.mean(axis=1)))


## Final submission

The submission must contain exactly the identifier column and the competition target, with one row per test example.


In [ ]:
final_test_prediction = np.clip(stack_test, 0.0, 1.0)

submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = final_test_prediction

assert len(submission) == len(test)
assert submission[ID_COL].equals(sample_submission[ID_COL])
assert submission[TARGET].notna().all()
assert submission[TARGET].between(0, 1).all()

submission_path = Path("/content/submission_s6e8_rank_gauss_stack.csv")
submission.to_csv(submission_path, index=False)

print("Submission saved to:", submission_path)
display(submission.head())
display(submission[TARGET].describe())


## Save experiment artifacts

These artifacts make the experiment auditable and make later blending experiments much easier. Keep them in Colab or Google Drive, not in the public repository unless they are intentionally small.


In [ ]:
ARTIFACT_DIR = Path("/content/s6e8_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

oof_frame.assign(target=y).to_csv(ARTIFACT_DIR / "oof_predictions.csv", index=False)
test_frame.to_csv(ARTIFACT_DIR / "test_predictions.csv", index=False)

all_scores = model_scores.copy()
all_scores.loc[len(all_scores)] = ["rank_average", roc_auc_score(y, oof_meta.mean(axis=1)), np.nan, np.nan]
all_scores.loc[len(all_scores)] = ["stack", stack_auc, np.nan, np.nan]
all_scores.to_csv(ARTIFACT_DIR / "model_scores.csv", index=False)

run_config = {
    "seed": SEED,
    "competition": COMPETITION,
    "target": TARGET,
    "n_folds": N_FOLDS,
    "n_iterations": N_ITERATIONS,
    "fast_mode": FAST_MODE,
    "run_xgb": RUN_XGB,
    "features": list(train_features.columns),
}
with open(ARTIFACT_DIR / "run_config.json", "w", encoding="utf-8") as file:
    json.dump(run_config, file, indent=2)

print("Artifacts saved to:", ARTIFACT_DIR)
print("Files:", sorted(path.name for path in ARTIFACT_DIR.iterdir()))


## Next experiments

For a serious Kaggle iteration, change one thing at a time and preserve the out-of-fold predictions:

1. Tune the feature-engineering formulas after checking their distributions.
2. Compare the stack with a simple weighted rank average.
3. Try CatBoost with and without the engineered ratios.
4. Tune LightGBM's leaves, minimum child samples, and regularization.
5. Test a second seed and average predictions only when the validation gain is stable.
6. Add a compact neural model only if its out-of-fold predictions are genuinely complementary.
7. Use local cross-validation as the primary decision tool; public leaderboard movement alone can be noisy.

The next submission should be uploaded from `/content/submission_s6e8_rank_gauss_stack.csv`.
